In [2]:
from typing import Literal, Optional
from pydantic import BaseModel, Field, ValidationError


# 1) Schema: the shape the LLM must return
class RateRequest(BaseModel):
    origin: str = Field(pattern=r"^[A-Z]{3}$", description="IATA code, e.g. FRA")
    destination: str = Field(pattern=r"^[A-Z]{3}$", description="IATA code, e.g. PVG")
    weight_kg: float = Field(gt=0)
    incoterm: Optional[Literal["EXW", "FCA", "DAP", "DDP"]] = None
    missing_fields: list[str] = []   # facts the email does not contain


# 2) Rate table: extracted once from an email attachment by an LLM, now plain data
RATES = [
    {"origin": "FRA", "destination": "PVG", "min_kg": 0,   "max_kg": 500,  "eur_per_kg": 3.10, "carrier": "LH"},
    {"origin": "FRA", "destination": "PVG", "min_kg": 501, "max_kg": 5000, "eur_per_kg": 2.60, "carrier": "LH"},
    {"origin": "FRA", "destination": "PVG", "min_kg": 0,   "max_kg": 5000, "eur_per_kg": 2.95, "carrier": "CX"},
    {"origin": "MUC", "destination": "JFK", "min_kg": 0,   "max_kg": 5000, "eur_per_kg": 2.20, "carrier": "LH"},
]


# 3) Lookup: plain code, no LLM
def find_rate(origin: str, destination: str, weight_kg: float):
    matches = [
        r for r in RATES
        if r["origin"] == origin
        and r["destination"] == destination
        and r["min_kg"] <= weight_kg <= r["max_kg"]
    ]
    return min(matches, key=lambda r: r["eur_per_kg"]) if matches else None


# The LLM read the email and returned this JSON text
llm_json = '{"origin": "FRA", "destination": "PVG", "weight_kg": 800, "incoterm": "EXW", "missing_fields": []}'

req = RateRequest.model_validate_json(llm_json)        # schema check
rate = find_rate(req.origin, req.destination, req.weight_kg)
print(rate)
print("cost EUR:", round(rate["eur_per_kg"] * req.weight_kg, 2))   # the cost is computed in code

# Bad LLM output is caught before it reaches the TMS
bad_json = '{"origin": "Frankfurt", "destination": "PVG", "weight_kg": -5}'
try:
    RateRequest.model_validate_json(bad_json)
except ValidationError as e:
    for err in e.errors():
        print("-", err["loc"][0], err["msg"])

{'origin': 'FRA', 'destination': 'PVG', 'min_kg': 501, 'max_kg': 5000, 'eur_per_kg': 2.6, 'carrier': 'LH'}
cost EUR: 2080.0
- origin String should match pattern '^[A-Z]{3}$'
- weight_kg Input should be greater than 0


In [3]:
tool = {
    "name": "find_rate",
    "description": "Find the cheapest rate for a lane and weight.",
    "input_schema": {                       # OpenAI calls this field "parameters"
        "type": "object",
        "properties": {
            "origin": {"type": "string"},
            "destination": {"type": "string"},
            "weight_kg": {"type": "number"},
        },
        "required": ["origin", "destination", "weight_kg"],
    },
}